# CSP Pipeline

This notebook loads the Layer E candidate slots, models the scheduling problem as a **Constraint Satisfaction Problem** (one variable per satellite), and splits it into independent sub-problems (connected components of the constraint graph). Each sub-problem is exported as a terminal subgraph for the shared annealing notebook.

The CSP formulation is intentionally different from the MWIS one (`02.A_mwis_pipeline.ipynb`): the two notebooks tackle the classical problem decomposition with two different approaches. The output artifacts, instead, follow the same format, so both producers can be compared downstream.

## 1. Configuration

All run parameters (input file, run name, QPU limits) are read from `results/_config.json` (or from the file in the `SWE2_CONFIG` environment variable). Set `run_name_csp` to select the CSP experiment and `csp_slot_csv` to select its input; the CSP graphics notebook uses the same settings. A copy of the configuration used is saved as `run_config_log.json` in `results/runs/csp/pipeline_results/<run_name_csp>`.

In [1]:
from __future__ import annotations

import gzip
import json
import os
import sys
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd
pd.set_option("display.max_columns", None)


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"

# project modules live in `src/`
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from CSP import CSP, CSP_Variable, Timeslot, split_connected_components


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS and QUBO notebooks)
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: candidate slots produced by the data-wrangling notebooks
CSP_SLOT_CSV = PROJECT_ROOT / run_config["csp_slot_csv"]

# CSP experiment name and output directory
RUN_NAME = run_config["run_name_csp"]
RUN_DIR = RESULTS_DIR / "runs" / "csp" / "pipeline_results" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# QPU physical constraints (the CSP does not decompose further yet: oversized sub-problems are only flagged)
MAX_QPU_TERMINAL_NODES = run_config["max_qpu_terminal_nodes"]
MAX_QPU_TERMINAL_COUPLERS = run_config["max_qpu_terminal_couplers"]

# logging environment configuration
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"CSP experiment: {RUN_NAME}")
print(f"Input dataset: {CSP_SLOT_CSV.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")

PHASE_PROFILE_PATH = RUN_DIR / f'csp_phase_profile_{time.strftime("%Y%m%d_%H%M%S")}.jsonl'
PIPELINE_STARTED = time.perf_counter()

def record_phase_timing(phase: str, started: float, **details: Any) -> float:
    """
    Helper function.
    Get how much time the specified phase takes to run, and store it in the log file.
    """
    elapsed = time.perf_counter() - started
    event = {"phase": phase, "elapsed_seconds": round(elapsed, 3), **details}
    with PHASE_PROFILE_PATH.open("a", encoding="utf-8") as profile_file:
        profile_file.write(json.dumps(event) + "\n")
        profile_file.flush()
    print(f"[CSP timing] {phase}: {elapsed:.2f}s", flush=True)
    return elapsed

Loaded configuration from: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\_config.json
CSP experiment: latest_csp_run
Input dataset: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv
Artifacts will be saved to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\csp\pipeline_results\latest_csp_run


## 2. Load candidate slots

The data-wrangling notebooks already created weighted candidate contacts: each raw visibility window was discretised into fixed-length candidate slots, one every `discretization_step_seconds`. Every slot inherits the priority, urgency and weight of its own parent window.

In [2]:
REQUIRED_COLUMNS = [
    "slot_id", "parent_window_id", "satellite_id", "satellite_name", "antenna_id", "station_id",
    "start_time_hours", "end_time_hours", "duration_hours",
    "reconfiguration_time_hours", "priority", "urgency", "weight",
]

raw_slots = pd.read_csv(CSP_SLOT_CSV)
slots = raw_slots[REQUIRED_COLUMNS].copy()

print({
    "candidate_slots": len(slots),
    "satellites": slots["satellite_name"].nunique(),
    "antennas": slots["antenna_id"].nunique(),
    "source_intervals": slots["parent_window_id"].nunique(),
    "time_horizon_hours": (round(slots["start_time_hours"].min(), 3), round(slots["end_time_hours"].max(), 3)),
})

{'candidate_slots': 2556, 'satellites': 95, 'antennas': 50, 'source_intervals': 2005, 'time_horizon_hours': (np.float64(0.0), np.float64(4.0))}


## 3. Translating to CSP

| Entity | CSP element |
|---|---|
| Satellite *S* | `CSP_Variable` (one per unique satellite) |
| Candidate slot at antenna *A* | `Timeslot` holding its `slot_id`, antenna, start/end and reconfiguration time (integer seconds) |
| All candidate slots of satellite *S* | Domain of variable *S* |
| Two satellites have a pair of slots on the same antenna that overlap, or leave less than the reconfiguration time between them (`Timeslot.overlap`) | *Constraint*: they cannot both take those slots |

Slot weights are not stored in the CSP (inference does not use them): they are looked up by `slot_id` when the conflict graph is built.

Antenna conflicts follow the same rule as the MWIS formulation. The model assumption that differs from MWIS is:
- each satellite receives **at most one** contact over the whole horizon.

In [3]:
phase_started = time.perf_counter()

# build the CSP: one variable per satellite, its domain is the slots it may use
csp = CSP(max_vars=slots["satellite_name"].nunique())
for sat_name, sat_rows in slots.groupby("satellite_name", sort=True):
    var = CSP_Variable(name=sat_name)
    for row in sat_rows.itertuples(index=False):
        var.add_value(Timeslot.from_hours(
            row.slot_id, row.antenna_id, row.reconfiguration_time_hours,
            row.start_time_hours, row.end_time_hours,
        ))
    # add_var links this variable to every satellite with a conflicting slot on the same antenna
    csp.add_var(var)

record_phase_timing("csp_build", phase_started, variables=len(csp.vars), constraints=csp.num_constraints())


# --- prettier summary print ---
def print_summary(label, summ):
    print(f'\n{"-"*46}')
    print(f'  {label}')
    print(f'{"-"*46}')
    for k, v in summ.items():
        val = f'{v:.4f}' if isinstance(v, float) else str(int(v))
        print(f'  {k:<24} {val}')

csp_summary = csp.compute_summary()
print_summary("CSP  —  summary", csp_summary)

[CSP timing] csp_build: 3.05s

----------------------------------------------
  CSP  —  BEFORE preprocessing
----------------------------------------------
  num_vars                 95
  avg_domain_size          26.9053
  sd_domain_size           17.3047
  min_domain_size          1.0000
  max_domain_size          78.0000
  num_const                600
  avg_degree               12.6316
  sd_degree                4.7241
  const_dens               0.1344


## 4. Connected-Component Decomposition

Satellites with no conflicting slots are completely independent: the constraint graph splits into connected components, each one a sub-problem that can be solved on its own and merged back at the end. The split is exact, since no constraint is cut.

In [ ]:
# satellites with no conflicting slots can never conflict: the constraint graph splits
# into independent sub-problems, each solvable on its own and merged back at the end.
phase_started = time.perf_counter()
# largest first, ties broken by the smallest satellite name (ids are local to each sub-CSP)
sorted_components = sorted(
    split_connected_components(csp),
    key=lambda c: (-len(c.vars), min(v.name for v in c.vars)),
)
record_phase_timing("connected_components", phase_started, component_count=len(sorted_components))

sizes = [len(c.vars) for c in sorted_components]
print(f"Connected components : {len(sorted_components)}")
print(f"  Sizes (# satellites)  : {sizes}")
print(f"  Largest component     : {sizes[0]} satellites")
print(f"  Singleton components  : {sum(1 for s in sizes if s == 1)}")

In [ ]:
# ----------------------------------------------------------
# Export the CSP structure for the graphics notebook
# ----------------------------------------------------------

# one row per satellite: domain size and degree
variables_rows = [
    {"var_id": v.id, "name": v.name, "domain_size": len(v.domain), "degree": csp.degree(v)}
    for v in csp.vars
]
pd.DataFrame(variables_rows).to_csv(RUN_DIR / "csp_variables.csv", index=False)

# one row per constraint (satellite pair with conflicting slots)
constraint_rows = [
    {"u_id": v.id, "v_id": u.id}
    for v in csp.vars
    for u in csp.neighbors(v)
    if u.id > v.id
]
pd.DataFrame(constraint_rows).to_csv(RUN_DIR / "csp_constraints.csv", index=False)

# one row per connected component (largest first)
components_rows = [
    {
        "component_id": f"component_{index:03d}",
        "num_vars": len(component.vars),
        "num_constraints": component.num_constraints(),
        "qubo_vars": sum(len(v.domain) for v in component.vars),
    }
    for index, component in enumerate(sorted_components, start=1)
]
pd.DataFrame(components_rows).to_csv(RUN_DIR / "csp_components.csv", index=False)

print(f"Exported CSP structure to: {RUN_DIR.resolve()}")

## 5. Hand-over to the QUBO notebook

Each sub-problem is exported as a **conflict graph**, the format `3_qubo_annealing.ipynb` reads: one node per candidate slot (`Timeslot`), identified by its `slot_id` and weighted by that slot's weight, and one edge per pair that cannot both be selected.

| CSP element | Conflict graph |
|---|---|
| Binary variable `x[t]` | Node (`slot_id`) |
| Reward for scheduling satellite *v* at *t* | Node weight (weight of that slot) |
| At most one slot per satellite | Edge between the nodes of the same satellite (`same satellite`) |
| Antenna conflict (`Timeslot.overlap`) | Edge between conflicting slots of different satellites (`same-antenna conflict`) |

An independent set of this graph is exactly a feasible assignment. As in the MWIS pipeline, isolated nodes (slots with no conflicts) are selected directly (`forced_selected`) and removed from the graph sent to the annealer.

The artifacts follow the MWIS pipeline format: `terminal_subgraphs.json`, `conflict_graph_nodes.csv` / `conflict_graph_edges.csv`, `components_summary.csv` and `reconstruction_metadata.json.gz`. Here every component is a single terminal: sub-problems larger than the QPU limits are flagged, not decomposed further.

In [ ]:
slot_attributes = slots.set_index("slot_id")


def component_to_conflict_graph(component: CSP) -> nx.Graph:
    """Turn one CSP component into a weighted conflict graph with one node per Timeslot."""
    graph = nx.Graph()

    for v in component.vars:
        var_nodes = []
        for t in sorted(v.domain, key=lambda t: t.slot_id):
            row = slot_attributes.loc[t.slot_id]
            graph.add_node(
                t.slot_id,
                source_interval_id=row.parent_window_id,
                satellite_id=str(row.satellite_id),
                satellite_name=v.name,
                antenna_id=row.antenna_id,
                station_id=row.station_id,
                start_time=float(row.start_time_hours),
                end_time=float(row.end_time_hours),
                duration=float(row.duration_hours),
                reconfiguration_time=float(row.reconfiguration_time_hours),
                priority=float(row.priority),
                urgency=float(row.urgency),
                weight=float(row.weight),
            )
            var_nodes.append(t.slot_id)

        # at most one slot per satellite
        for u, w in combinations(var_nodes, 2):
            graph.add_edge(u, w, reason="same satellite")

    # antenna conflict: slots of constrained satellites that overlap (or are closer than the reconfiguration time)
    for v in component.vars:
        for u in component.neighbors(v):
            if u.id <= v.id:
                continue
            for t1 in v.domain:
                for t2 in u.domain:
                    if Timeslot.overlap(t1, t2):
                        graph.add_edge(t1.slot_id, t2.slot_id, reason="same-antenna conflict")

    return graph


def component_summary(component_id: str, graph: nx.Graph) -> dict[str, Any]:
    weights = [data["weight"] for _, data in graph.nodes(data=True)]
    return {
        "component_id": component_id,
        "node_count": graph.number_of_nodes(),
        "edge_count": graph.number_of_edges(),
        "density": nx.density(graph) if graph.number_of_nodes() > 1 else 0.0,
        "total_weight": sum(weights),
        "max_weight": max(weights, default=0.0),
        "satellite_count": len({data["satellite_id"] for _, data in graph.nodes(data=True)}),
        "antenna_count": len({data["antenna_id"] for _, data in graph.nodes(data=True)}),
    }


def graph_fits_qpu_limits(graph: nx.Graph) -> bool:
    """
    Return whether both terminal-size limits are satisfied.
    """
    return (
        graph.number_of_nodes() <= MAX_QPU_TERMINAL_NODES
        and graph.number_of_edges() <= MAX_QPU_TERMINAL_COUPLERS
    )


# Convert a graph to plain JSON: nodes and edges with their attributes
def serialize_graph(graph: nx.Graph) -> dict[str, Any]:
    return {
        "nodes": [{"id": node, **data} for node, data in graph.nodes(data=True)],
        "edges": [{"source": u, "target": v, **data} for u, v, data in graph.edges(data=True)],
    }

In [ ]:
phase_started = time.perf_counter()
components = []             # one record per component: id, full conflict graph, summary
terminal_subgraphs = []     # non-empty terminals, read by the annealing notebook
metadata_nodes = []         # one terminal node per component, read by reconstruction tools

for index, component in enumerate(sorted_components, start=1):
    component_id = f"component_{index:03d}"
    graph = component_to_conflict_graph(component)
    components.append({"component_id": component_id, "graph": graph, "summary": component_summary(component_id, graph)})

    # isolated slots have no conflict: select them directly, as the MWIS preprocessing does
    forced_selected = sorted(nx.isolates(graph))
    forced_objective = sum(graph.nodes[node]["weight"] for node in forced_selected)
    reduced = graph.subgraph(set(graph.nodes) - set(forced_selected))
    terminal_reason = (
        "resolved by exact preprocessing" if reduced.number_of_nodes() == 0
        else "fits QPU limits" if graph_fits_qpu_limits(reduced)
        else "exceeds QPU limits (no further CSP decomposition)"
    )
    terminal_id = f"{component_id}_dnode_0001"

    if reduced.number_of_nodes() > 0:
        terminal_subgraphs.append({
            "terminal_id": terminal_id,
            "original_component_id": component_id,
            "depth": 0,
            "node_count": reduced.number_of_nodes(),
            "edge_count": reduced.number_of_edges(),
            "forced_objective": forced_objective,
            "forced_selected": forced_selected,
            "terminal_reason": terminal_reason,
            "graph": serialize_graph(reduced),
        })

    metadata_nodes.append({
        "node_id": terminal_id,
        "component_id": component_id,
        "node_type": "terminal",
        "depth": 0,
        "input_node_count": graph.number_of_nodes(),
        "input_edge_count": graph.number_of_edges(),
        "qpu_node_count": reduced.number_of_nodes(),
        "qpu_edge_count": reduced.number_of_edges(),
        "forced_selected": forced_selected,
        "forced_objective": float(forced_objective),
        "terminal_reason": terminal_reason,
        "terminal_id": terminal_id if reduced.number_of_nodes() else None,
    })

# full conflict graph (union of the components) for inspection and downstream notebooks
conflict_graph = nx.union_all([component["graph"] for component in components])
node_rows = [{"slot_id": node_id, **data} for node_id, data in conflict_graph.nodes(data=True)]
edge_rows = [{"source": u, "target": v, **data} for u, v, data in conflict_graph.edges(data=True)]
pd.DataFrame(node_rows).to_csv(RUN_DIR / "conflict_graph_nodes.csv", index=False)
pd.DataFrame(edge_rows).to_csv(RUN_DIR / "conflict_graph_edges.csv", index=False)

components_summary = pd.DataFrame([component["summary"] for component in components])
components_summary.to_csv(RUN_DIR / "components_summary.csv", index=False)

(RUN_DIR / "terminal_subgraphs.json").write_text(json.dumps(terminal_subgraphs, indent=2), encoding="utf-8")

# Reconstruction map: one terminal root per component (no separator decomposition in the CSP pipeline)
reconstruction_metadata = {
    "schema_version": 1,
    "producer": "csp_pipeline",
    "run_name": RUN_NAME,
    "objective": "maximum-weight assignment, at most one timeslot per satellite",
    "terminal_subgraphs_file": "terminal_subgraphs.json",
    "components": [
        {"component_id": node["component_id"], "root_node_id": node["node_id"], "status": "complete"}
        for node in metadata_nodes
    ],
    "nodes": metadata_nodes,
}
with gzip.open(RUN_DIR / "reconstruction_metadata.json.gz", "wt", encoding="utf-8", compresslevel=6) as metadata_file:
    json.dump(reconstruction_metadata, metadata_file)

oversized = [t for t in terminal_subgraphs if t["terminal_reason"].startswith("exceeds")]
record_phase_timing("conflict_graph_and_terminal_export", phase_started, terminal_subgraphs=len(terminal_subgraphs))

print({
    "nodes": conflict_graph.number_of_nodes(),
    "edges_couplers": conflict_graph.number_of_edges(),
    "edge_reasons": dict(Counter(data["reason"] for _, _, data in conflict_graph.edges(data=True))),
    "components": len(components),
    "terminal_subgraphs": len(terminal_subgraphs),
    "terminals_exceeding_qpu_limits": len(oversized),
    "largest_terminal_nodes": max((t["node_count"] for t in terminal_subgraphs), default=0),
    "largest_terminal_couplers": max((t["edge_count"] for t in terminal_subgraphs), default=0),
    "qpu_limits": (MAX_QPU_TERMINAL_NODES, MAX_QPU_TERMINAL_COUPLERS),
})
components_summary.head(10)

## 6. Final summary table

This table is the main compact output of the notebook. The graphics and QUBO notebooks read the artifacts saved above instead of rebuilding this pipeline.

In [ ]:
domain_values = sum(len(v.domain) for v in csp.vars)

# Final metrics as (name, value, meaning)
final_summary_rows = [
    ("Candidate slots", len(slots), "Rows of the input dataset."),
    ("Satellites (CSP variables)", len(csp.vars), "One variable per satellite."),
    ("Domain values", domain_values, "Candidate slots in the domains, i.e. QUBO variables."),
    ("Constraints", int(csp_summary["num_const"]), "Satellite pairs with at least one conflicting slot pair."),
    ("Number of components (total)", len(components), "Connected components of the constraint graph."),
    ("Number of couplers (total)", conflict_graph.number_of_edges(), "Conflict edges in the full graph."),
    ("Terminal subgraphs", len(terminal_subgraphs), "Non-empty sub-problems exported for annealing."),
    ("Terminal subgraphs exceeding QPU limits", len(oversized), "Sub-problems that would need a further decomposition."),
    ("Forced-selected objective", sum(node["forced_objective"] for node in metadata_nodes), "Weight of isolated slots selected without annealing."),
]

final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value", "meaning"])
phase_started = time.perf_counter()
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)
record_phase_timing("final_reporting_export", phase_started, metrics=len(final_summary))
record_phase_timing("pipeline_total_so_far", PIPELINE_STARTED, run_name=RUN_NAME)

# Report the shared annealing target without changing it (it may intentionally target MWIS).
print(f"CSP experiment saved: {RUN_NAME}")
annealing_target = run_config.get("qubo_input_run")
if annealing_target and (RESULTS_DIR / annealing_target).resolve() == RUN_DIR.resolve():
    print("The shared annealing input matches this CSP experiment.")
else:
    print(f"The shared annealing notebook currently targets: {annealing_target or '(not configured)'}")
    print("To anneal this CSP experiment, set qubo_input_run to:")
    print(RUN_DIR.relative_to(RESULTS_DIR).as_posix())

final_summary